## **Northeastern SMILE Lab - Recognizing Faces in the Wild**

## Background

The SMILE Lab at Northeastern focuses on applied machine learning, social media analytics, human-computer interaction, and image/video understanding. This notebook applies that domain to the Families in the Wild (FIW) kinship-verification task.

Kinship recognition is challenging because available datasets may not fully represent global familial diversity, and facial similarity reflects both genetic and environmental factors. A learned image-pair model is therefore better suited than a fixed rule-based visual comparison.

## Objective

Using the provided competition data, this notebook predicts whether two face images depict related individuals. Throughout the notebook, `1` means related and `0` means unrelated.

## Notebook scope

This repository includes two related notebooks:

- `main.ipynb`: local/full workflow that downloads Kaggle data, trains and evaluates the model, and generates a Kaggle submission file.
- `kaggle.ipynb`: Kaggle Notebook adaptation that assumes the competition dataset is mounted under `/kaggle/input` and focuses on training plus held-out evaluation.

## Roadmap

The workflow downloads and extracts the competition data, cleans labeled relationship rows, builds balanced image pairs with a family-aware split, trains a Siamese face-embedding model, evaluates it on a held-out labeled test split, and finally generates predictions for the unlabeled Kaggle submission images.


## **Step 1**: Download the official Kaggle competition (FIW) data.

Before executing the following cells, review section 7 of the Kaggle competition rules: https://www.kaggle.com/competitions/recognizing-faces-in-the-wild/rules#7-competition-data.

Prerequisites for the local workflow:

- A Kaggle account with the FIW competition rules accepted.
- A configured Kaggle API token for local downloads.
- Local storage for the downloaded zip files and extracted `_provided-data`, `_train-faces`, and `_test-faces` directories.

FIW data usage should cite the official FIW/RFIW papers:

- Joseph P. Robinson, Ming Shao, Hongfu Liu, Yue Wu, Timothy Gillis, and Yun Fu. "Visual Kinship Recognition of Families In the Wild." IEEE TPAMI Special Edition: Computational Face, 2018.
- Joseph P. Robinson, Ming Shao, Handong Zhao, Yue Wu, Timothy Gillis, and Yun Fu. "Recognizing Families In the Wild (RFIW): Data Challenge Workshop in conjunction with ACM MM 2017." ACM Multimedia Conference: Workshop on RFIW, 2017.
- Shuyang Wang, Joseph P. Robinson, and Yun Fu. "Kinship Verification on Families in the Wild with Marginalized Denoising Metric Learning." IEEE Automatic Face and Gesture Recognition, 2017.
- Joseph P. Robinson, Ming Shao, Yue Wu, and Yun Fu. "Families In the Wild (FIW): large-scale kinship image database and benchmarks." ACM Multimedia Conference, 2016.


In [1]:
download_path = '_provided-data' # Intermediate directories are excluded recursively via `.gitignore` (i.e., '_*/').
competition = 'recognizing-faces-in-the-wild' # Hosted here: https://www.kaggle.com/c/recognizing-faces-in-the-wild


In [ ]:
import os
from kaggle.api.kaggle_api_extended import KaggleApi

zip_path = download_path + '/' + competition + '.zip'
if os.path.exists(zip_path):
    print('Data already downloaded: ' + zip_path)
else:
    os.makedirs(download_path, exist_ok=True)
    print('Downloading ' + competition + ' provided data into ' + download_path)
    api = KaggleApi()
    api.authenticate()
    api.competition_download_files(competition, path = download_path)

print('Step 1a complete: Kaggle data downloaded.')

In [ ]:
import os
import zipfile

def unzip(zip_path):
    dest_dir = '_' + os.path.basename(zip_path)[:-4]
    if (not os.path.exists(dest_dir)):
        os.makedirs(dest_dir, exist_ok=True)
        print('Decompressing ' + zip_path + ' into ' + dest_dir)
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(dest_dir)
    print(dest_dir + '/')
    print(os.listdir(dest_dir)[:10])
    return dest_dir

print('Decompressing ' + competition + ' provided data:')
dataset_path = unzip(download_path + '/' + competition + '.zip')
training_image_path = unzip(dataset_path + '/train-faces.zip')
testing_image_path = unzip(dataset_path + '/test-faces.zip')

print('Step 1b complete: data decompressed.')

## **Step 2**: Load and clean the labeled dataset.

The `train_relationships.csv` file encodes ground-truth kinship relationships between pairs of people. The `p1` and `p2` columns use a `family/member` path format, which is later used to find each member's face images under the extracted training-image directory.

Rows are removed when either member does not have corresponding image data. This keeps later pair generation from failing on missing folders or images.


In [4]:
from collections import defaultdict
import glob
import pandas

relations_csv_path = dataset_path + '/train_relationships.csv'
print('Loading ' + relations_csv_path)
relations_df = pandas.read_csv(relations_csv_path, delimiter=',', header='infer')

# Create a dictionary to lookup image files for each member
family_dict = defaultdict(list)
for family in glob.glob(training_image_path + '*'):
    for member in glob.glob(family + '/*'):
        for image_path in glob.glob(member + '/*'):
            member = os.path.basename(member)
            image_path = os.path.basename(image_path)
            family_dict[member].append(image_path)

print('Images found for members of ' + str(len(family_dict.items())) + ' total families.')
for key, value in list(family_dict.items())[:5]:
    print(str(key) + ': ' + str(value) + ',')
    
# Remove entries which do not exist in the training set
print('The original ground truth relations data contains ' + str(len(relations_df)) + ' pairs.')
print(relations_df)

print('Checking for missing relation image data.')
fam_keys = family_dict.keys()
missing_relations_list = []
for index, row in relations_df.iterrows():
    split1 = row.p1.split('/')
    split2 = row.p2.split('/')
    p1fam = split1[0]
    p2fam = split2[0]
    if (p1fam not in fam_keys or p2fam not in fam_keys):
        missing_relations_list.append(index)
        continue
    p1member = split1[1]
    p2member = split2[1]
    if (p1member not in family_dict[p1fam] or p2member not in family_dict[p2fam]):
        missing_relations_list.append(index)
        continue
    images1 = os.listdir(training_image_path + '/' + p1fam + '/' + p1member)
    images2 = os.listdir(training_image_path + '/' + p2fam + '/' + p2member)
    if (len(images1) == 0 or len(images2) == 0):
        missing_relations_list.append(index)
        continue
if missing_relations_list:
    relations_df = relations_df.drop(missing_relations_list)
    print(str(len(missing_relations_list)) + ' pairs were removed due to missing data.')
    print(relations_df)
else:
    print('All relations were found to have valid image data.')

Loading _recognizing-faces-in-the-wild/train_relationships.csv
Images found for members of 786 total families.
F0658: ['MID6', 'MID1', 'MID7', 'MID2', 'MID5', 'MID3'],
F0667: ['MID1', 'MID2'],
F0031: ['MID1', 'MID2', 'MID5', 'MID4', 'MID3'],
F0693: ['MID1', 'MID2', 'MID4', 'MID3'],
F0499: ['MID1', 'MID2', 'MID3'],
The original ground truth relations data contains 3598 pairs.
              p1          p2
0     F0002/MID1  F0002/MID3
1     F0002/MID2  F0002/MID3
2     F0005/MID1  F0005/MID2
3     F0005/MID3  F0005/MID2
4     F0009/MID1  F0009/MID4
...          ...         ...
3593  F1000/MID5  F1000/MID8
3594  F1000/MID5  F1000/MID9
3595  F1000/MID6  F1000/MID9
3596  F1000/MID7  F1000/MID8
3597  F1000/MID7  F1000/MID9

[3598 rows x 2 columns]
Checking for missing relation image data.
1108 pairs were removed due to missing data.
              p1          p2
0     F0002/MID1  F0002/MID3
1     F0002/MID2  F0002/MID3
2     F0005/MID1  F0005/MID2
3     F0005/MID3  F0005/MID2
29    F0016/MID1 

## **Step 3**: Show a random selection of known related person pairs.

This visual check samples labeled relationships from the cleaned CSV and displays one image for each person in the pair.


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
from random import choice

SAMPLE_COUNT = 3
samples = relations_df.sample(SAMPLE_COUNT)
print(samples)

f, ax = plt.subplots(SAMPLE_COUNT, 2)
for a in ax.flat:
    a.axis('off')
i = 0
for member in samples.values:
    img1 = training_image_path + '/' + member[0] + '/' \
        + choice(os.listdir(training_image_path + '/' + member[0]))
    img2 = training_image_path + '/' + member[1] + '/'  \
        + choice(os.listdir(training_image_path + '/' + member[1]))
    rgb1 = Image.open(img1)
    rgb2 = Image.open(img2)
    ax[i][0].imshow(rgb1)
    ax[i][1].imshow(rgb2)
    i = i + 1

print('Step 3 complete: sample relations displayed.')

## **Step 4**: Split by family and generate balanced train, validation, and held-out test pairs.

Pairs are split by family so that members of the same family do not appear across training, validation, and held-out test splits. This reduces leakage: the model should be evaluated on families it did not see during training.

Positive pairs come from labeled kinship relationships. Negative pairs are sampled from members without a listed relationship and are treated as presumed unrelated for training balance; they are not independently verified non-kin pairs.


In [6]:
import itertools
import random
import numpy
import torch

random.seed(42)
numpy.random.seed(42)
torch.manual_seed(42)

# Extract unique family IDs and split 70/15/15
all_families = sorted(set(row.p1.split('/')[0] for _, row in relations_df.iterrows()))
random.shuffle(all_families)
n = len(all_families)
train_cutoff = int(0.70 * n)
val_cutoff = int(0.85 * n)
train_families = set(all_families[:train_cutoff])
val_families = set(all_families[train_cutoff:val_cutoff])
test_families = set(all_families[val_cutoff:])
print(f'Family split: {len(train_families)} train, {len(val_families)} val, {len(test_families)} test')

# Build member image lookup from all members in relations_df
members = sorted(set(m for row in relations_df.values for m in row))
member_images = dict()
for member in members:
    image_files = os.listdir(training_image_path + '/' + member)
    if len(image_files) > 0:
        member_images[member] = [member + '/' + img for img in image_files]

def generate_pairs(relations_subset, family_set):
    """Generate balanced positive and negative pairs for a family split."""
    # Positive pairs: all image combinations for each related pair
    positives = []
    for _, row in relations_subset.iterrows():
        p1_images = member_images.get(row.p1, [])
        p2_images = member_images.get(row.p2, [])
        for img1, img2 in itertools.product(p1_images, p2_images):
            positives.append([img1, img2, 1.0])

    # Negative pairs: sample unrelated members within the same family set
    split_members = [m for m in members if m.split('/')[0] in family_set and m in member_images]
    negatives = []
    while len(negatives) < len(positives):
        p1 = random.choice(split_members)
        p2 = random.choice(split_members)
        if p1 == p2:
            continue
        p1_images = member_images[p1]
        p2_images = member_images[p2]
        for img1, img2 in itertools.product(p1_images, p2_images):
            negatives.append([img1, img2, 0.0])
            if len(negatives) >= len(positives):
                break

    data = positives + negatives[:len(positives)]
    random.shuffle(data)
    return data, len(positives), len(negatives[:len(positives)])

# Split relations by family
train_relations = relations_df[relations_df['p1'].apply(lambda x: x.split('/')[0] in train_families)]
val_relations = relations_df[relations_df['p1'].apply(lambda x: x.split('/')[0] in val_families)]
test_relations = relations_df[relations_df['p1'].apply(lambda x: x.split('/')[0] in test_families)]

training_data, train_pos, train_neg = generate_pairs(train_relations, train_families)
val_data, val_pos, val_neg = generate_pairs(val_relations, val_families)
testing_data, test_pos, test_neg = generate_pairs(test_relations, test_families)

print(f'Training:   {len(training_data)} pairs ({train_pos} pos + {train_neg} neg)')
print(f'Validation: {len(val_data)} pairs ({val_pos} pos + {val_neg} neg)')
print(f'Testing:    {len(testing_data)} pairs ({test_pos} pos + {test_neg} neg)')

Family split: 249 train, 53 val, 54 test
Training:   216018 pairs (108009 pos + 108009 neg)
Validation: 17362 pairs (8681 pos + 8681 neg)
Testing:    23776 pairs (11888 pos + 11888 neg)


## **Step 5**: Train the model.

This section creates image-pair datasets, defines the Siamese network, trains it with contrastive loss, runs a training-set sanity check, and saves the trained weights.

The model uses a Siamese architecture: the same pretrained `InceptionResnetV1` face encoder processes both images, producing embeddings in a shared space. Contrastive loss pulls related pairs closer together and pushes presumed unrelated pairs apart. During evaluation, lower embedding distance means stronger predicted kinship similarity.


In [7]:
import torch
from torch.utils.data import Dataset
import torchvision.transforms as transforms
from PIL import Image

# Training transform with data augmentation.
train_transform = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

# Evaluation transform without random augmentations.
eval_transform = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

class ImagePairDataset(Dataset):
    def __init__(self, data, transform):
        self.image_pairs = [sublist[:-1] for sublist in data]
        self.labels = torch.tensor([sublist[-1] for sublist in data], dtype=torch.float32)
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img0 = Image.open('_train-faces/' + self.image_pairs[idx][0])
        img1 = Image.open('_train-faces/' + self.image_pairs[idx][1])
        img0 = self.transform(img0)
        img1 = self.transform(img1)
        return img0, img1, self.labels[idx]

batch_size = 64

train_dataset = ImagePairDataset(training_data, train_transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
print(f'Train: {len(train_dataset)} samples, {len(train_loader)} batches')

val_dataset = ImagePairDataset(val_data, eval_transform)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
print(f'Val:   {len(val_dataset)} samples, {len(val_loader)} batches')

test_dataset = ImagePairDataset(testing_data, eval_transform)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
print(f'Test:  {len(test_dataset)} samples, {len(test_loader)} batches')

Train: 216018 samples, 3376 batches
Val:   17362 samples, 272 batches
Test:  23776 samples, 372 batches


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from facenet_pytorch import InceptionResnetV1

class SiameseNetwork(nn.Module):
    def __init__(self):
        super(SiameseNetwork, self).__init__()

        # Pretrained FaceNet backbone (#7)
        backbone = InceptionResnetV1(pretrained='vggface2')

        # Freeze all layers
        for param in backbone.parameters():
            param.requires_grad = False

        # Unfreeze last 2 blocks (repeat_3 and block8)
        for name, param in backbone.named_parameters():
            if name.startswith('repeat_3.') or name.startswith('block8.'):
                param.requires_grad = True

        # Use backbone up through avgpool (strip the classification head)
        self.backbone = nn.Sequential(
            backbone.conv2d_1a,
            backbone.conv2d_2a,
            backbone.conv2d_2b,
            backbone.maxpool_3a,
            backbone.conv2d_3b,
            backbone.conv2d_4a,
            backbone.conv2d_4b,
            backbone.repeat_1,
            backbone.mixed_6a,
            backbone.repeat_2,
            backbone.mixed_7a,
            backbone.repeat_3,
            backbone.block8,
            backbone.avgpool_1a,
            nn.Flatten(),
            backbone.dropout,
            backbone.last_linear,
            backbone.last_bn,
        )

        # FC head: 512 → 128 embedding
        self.fc1 = nn.Linear(512, 128)

    def forward_once(self, x):
        output = self.backbone(x)
        output = self.fc1(output)
        return output

    def forward(self, input1, input2):
        output1 = self.forward_once(input1)
        output2 = self.forward_once(input2)
        return output1, output2

class ContrastiveLoss(torch.nn.Module):
    def __init__(self, margin=2.0):
        super(ContrastiveLoss, self).__init__()
        self.margin = margin

    def forward(self, output1, output2, label):
        euclidean_distance = F.pairwise_distance(output1, output2, keepdim = True)
        # label=1 means related (similar) → minimize distance
        # label=0 means unrelated (dissimilar) → push distance beyond margin
        loss_contrastive = torch.mean((label) * torch.pow(euclidean_distance, 2) +
                                      (1-label) * torch.pow(torch.clamp(self.margin - euclidean_distance, min=0.0), 2))
        return loss_contrastive

# Hyperparameters (reduced epochs — pretrained backbone converges faster)
num_epochs = 10
learning_rate = 0.0001

# Device selection: prefer CUDA, then MPS (Apple Silicon GPU), then CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# Create the network
model = SiameseNetwork().to(device)
criterion = ContrastiveLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=learning_rate)

print(f'Step 5a complete: model initialized on device: {device}')

In [ ]:
def train(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0

    for i, (data1, data2, label) in enumerate(dataloader):
        data1, data2, label = data1.to(device), data2.to(device), label.to(device)
        optimizer.zero_grad()
        output1, output2 = model(data1, data2)
        loss = criterion(output1, output2, label)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * data1.size(0)

    return running_loss / len(dataloader.dataset)

def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for data1, data2, label in dataloader:
            data1, data2, label = data1.to(device), data2.to(device), label.to(device)
            output1, output2 = model(data1, data2)
            loss = criterion(output1, output2, label)
            running_loss += loss.item() * data1.size(0)
    return running_loss / len(dataloader.dataset)

for epoch in range(num_epochs):
    train_loss = train(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, val_loader, criterion, device)
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:3d}/{num_epochs} — Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')

print('Finished training.')

In [ ]:
# Training-set sanity check: related pairs should have smaller distances than unrelated pairs.
model.eval()
pos_distances = []
neg_distances = []

with torch.no_grad():
    for data1, data2, label in train_loader:
        data1, data2, label = data1.to(device), data2.to(device), label.to(device)
        output1, output2 = model(data1, data2)
        distances = F.pairwise_distance(output1, output2)
        for d, l in zip(distances, label):
            if l.item() == 1.0:
                pos_distances.append(d.item())
            else:
                neg_distances.append(d.item())

mean_pos = sum(pos_distances) / len(pos_distances)
mean_neg = sum(neg_distances) / len(neg_distances)
print(f'Mean distance (related pairs):   {mean_pos:.4f}')
print(f'Mean distance (unrelated pairs): {mean_neg:.4f}')
if mean_pos < mean_neg:
    print('PASS: Related pairs are closer than unrelated pairs.')
else:
    print('FAIL: Related pairs are NOT closer. The model may not be learning meaningful representations.')

In [ ]:
# Store the trained model weights to file.
torch.save(model.state_dict(),'_model-state-dict.pth')
print('Step 5c complete: model saved to _model-state-dict.pth')

## **Step 6**: Evaluate on the held-out labeled test split.

This evaluates the model on the internal held-out labeled test split created from `train_relationships.csv`. This is separate from the unlabeled Kaggle competition `test-faces` submission set used in the next step.


In [ ]:
# Load the trained model weights from file.
model.load_state_dict(torch.load('_model-state-dict.pth', map_location=device, weights_only=True))
model.to(device)
print(f'Step 6a complete: model loaded from _model-state-dict.pth onto device: {device}')

In [ ]:
# Test DataLoader is already created in the dataset setup cell (test_loader).
# Proceeding directly to evaluation.

In [ ]:
def test(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_distances = []
    all_labels = []

    with torch.no_grad():
        for i, (data1, data2, label) in enumerate(dataloader):
            data1, data2, label = data1.to(device), data2.to(device), label.to(device)

            output1, output2 = model(data1, data2)
            loss = criterion(output1, output2, label)
            running_loss += loss.item() * data1.size(0)

            distances = F.pairwise_distance(output1, output2)
            all_distances.extend(distances.cpu().numpy())
            all_labels.extend(label.cpu().numpy())

    epoch_loss = running_loss / len(dataloader.dataset)
    print(f'Test Loss: {epoch_loss:.4f}')
    return all_distances, all_labels

test_distances, test_labels = test(model, test_loader, criterion, device)
print('Step 6b complete: test evaluation finished.')

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, precision_score, recall_score

distances = np.array(test_distances)
labels = np.array(test_labels)

# Convert distances to similarity scores (lower distance = higher similarity)
scores = 1 - (distances / distances.max())

# AUC-ROC
auc = roc_auc_score(labels, scores)
print(f'AUC-ROC: {auc:.4f}')

# ROC curve
fpr, tpr, thresholds = roc_curve(labels, scores)

# Optimal threshold via Youden's J statistic
j_scores = tpr - fpr
optimal_idx = np.argmax(j_scores)
optimal_threshold = thresholds[optimal_idx]
print(f'Optimal threshold (Youden\'s J): {optimal_threshold:.4f}')

# Classification metrics at optimal threshold
predictions = (scores >= optimal_threshold).astype(float)
acc = accuracy_score(labels, predictions)
prec = precision_score(labels, predictions)
rec = recall_score(labels, predictions)
print(f'Accuracy:  {acc:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall:    {rec:.4f}')

# Plot ROC curve
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(fpr, tpr, label=f'ROC (AUC = {auc:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', label='Random')
axes[0].scatter(fpr[optimal_idx], tpr[optimal_idx], color='red', zorder=5, label=f'Optimal ({optimal_threshold:.2f})')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()

# Distance distributions
pos_dist = distances[labels == 1.0]
neg_dist = distances[labels == 0.0]
axes[1].hist(pos_dist, bins=30, alpha=0.6, label=f'Related (n={len(pos_dist)})')
axes[1].hist(neg_dist, bins=30, alpha=0.6, label=f'Unrelated (n={len(neg_dist)})')
axes[1].set_xlabel('Euclidean Distance')
axes[1].set_ylabel('Count')
axes[1].set_title('Distance Distributions')
axes[1].legend()

plt.tight_layout()
plt.show()

print('Step 6c complete: test metrics and plots generated.')

## **Step 7**: Generate Kaggle submission predictions.

Kaggle submissions are evaluated on area under the ROC curve between the submitted `is_related` values and the hidden target labels. Not all pairs will be scored.

The submitted `is_related` value is derived from embedding distance: lower distance is converted into a higher kinship similarity score. This score is useful for ranking pairs for AUC evaluation, but it is not calibrated as a true probability unless an additional calibration step is added.

The Kaggle competition test-faces submission set is unlabeled. The cells below preprocess those images, score all requested image pairs, and write `submission.csv`.

**Submission file**

For each `img_pair` in the test set, predict a value for `is_related`. The `img_pair` column describes the pair of images; for example, `abcdef-ghijkl` means the pair `abcdef.jpg` and `ghijkl.jpg`.

The file should contain a header and have the following format:

```csv
img_pair,is_related
X3Nk6Hfe5x-qcZrTXsfde,0.0
X3Nk6Hfe5x-LD0pWDM8w_,0.0
X3Nk6Hfe5x-PHwuDtHyGp,0.0
X3Nk6Hfe5x-LO6lN_U4ot,0.0
...
```


The following preprocessing step builds an in-memory lookup table for the unlabeled Kaggle test images and creates all image combinations to score. Generating all combinations can be computationally expensive for large test sets.


In [ ]:

from PIL import Image
import itertools
import os

def pre_process(filename):
    img = Image.open(filename)
    img = eval_transform(img)
    img = img.unsqueeze(0)
    return img.to(device)

# Create a lookup table of pre-processed test images (processed_image_tensors).
processed_image_tensors = {}
basenames = [os.path.splitext(file)[0] for file in os.listdir('_test-faces')]
for basename in basenames:
    processed_image_tensors[basename] = pre_process(testing_image_path + '/' + basename + '.jpg')
print('There are ' + str(len(processed_image_tensors)) + ' total entries in the lookup table.')

# Generate a list of all pairs of test images.
test_pairs = list(itertools.combinations(basenames, 2))
print('There are ' + str(len(test_pairs)) + ' total pairs in the test set.')

print('Step 7a complete: test images pre-processed.')

In [ ]:
import csv
import torch
import torch.nn.functional as F

def is_related(img1, img2, model):
    with torch.no_grad():
        output1, output2 = model(img1, img2)
    euclidean_distance = F.pairwise_distance(output1, output2, keepdim = True)
    probability = 1 - torch.sigmoid(euclidean_distance).item()
    return probability

filename = "submission.csv"
with open(filename, 'w', newline='') as file:
    csv_writer = csv.writer(file)
    csv_writer.writerows([['img_pair', 'is_related']])

# Evaluate the test image pairs
model.eval()  # Set model to evaluation mode
# Results are written in blocks to enable intermediate inspection.
i = 0
block_size = 10000
results_block = []
random.shuffle(test_pairs)
for pair in list(test_pairs):
    images = list(pair)
    results_block.append([images[0] + '-' + images[1],
                          format(is_related(processed_image_tensors[images[0]],
                                            processed_image_tensors[images[1]],
                                            model), '.6f'),
                          ])
    i = i + 1
    if (0 == i % block_size):
        with open(filename, 'a', newline='') as file:
            csv_writer = csv.writer(file)
            csv_writer.writerows(results_block)
        results_block = []

with open(filename, 'a', newline='') as file:
    csv_writer = csv.writer(file)
    csv_writer.writerows(results_block)

print(f'Step 7b complete: {len(test_pairs)} pair predictions written to {filename}.')